In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Reviews.csv')

# Check shape and column types
print(df.shape)
print(df.dtypes)
print(df[['ProductId', 'Score', 'Summary', 'Text']].head(3))

# Check for missing values
print(df[['Text', 'Summary', 'Score']].isnull().sum())

# Check score distribution
print(df['Score'].value_counts())


In [ ]:
# Score distribution sorted by value
print(df[['Text', 'Summary', 'Score']].isnull().sum())
print('\nScore distribution:')
print(df['Score'].value_counts().sort_index())

# Number of unique products
print(f'\nUnique products: {df["ProductId"].nunique()}')

# Sample review text
print('\nSample review:')
print(df['Text'].iloc[0][:300])


In [ ]:
import pandas as pd

# Reload data to start fresh
df = pd.read_csv('Reviews.csv')

# Fill missing summaries with empty string
df['Summary'] = df['Summary'].fillna('')


def create_chunk(row):
    """Build a text chunk for each review with product info and sentiment label."""
    if row['Score'] >= 4:
        sentiment = 'positive'
    elif row['Score'] <= 2:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'
    return (
        f"Product: {row['ProductId']}\n"
        f"Score: {row['Score']}/5 ({sentiment})\n"
        f"Title: {row['Summary']}\n"
        f"Review: {row['Text'][:500]}"
    )


# Balanced sampling: 2000 reviews per score (1 to 5)
# Avoids bias toward 5-star reviews which dominate the dataset
# Note: pandas 3.x removed include_groups, so we use pd.concat instead of groupby.apply
df_sample = pd.concat([
    group.sample(min(len(group), 2000), random_state=42)
    for _, group in df.groupby('Score')
]).reset_index(drop=True)

# Create one chunk string per review
df_sample['chunk'] = df_sample.apply(create_chunk, axis=1)

print(f'Total chunks: {len(df_sample)}')
print(f'\nScore distribution (sample):')
print(df_sample['Score'].value_counts().sort_index())
print(f'\nSample chunk:\n{df_sample["chunk"].iloc[0]}')

# Save as a plain list for later steps
chunks = df_sample['chunk'].tolist()


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

# Load the embedding model
# all-MiniLM-L6-v2 is fast and works well for English text
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded: {model.get_sentence_embedding_dimension()} dimensions')

# Encode all chunks into vectors (~1-2 min for 10,000 chunks)
print('Creating embeddings...')
embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f'Embeddings shape: {embeddings.shape}')

# Normalize vectors so inner product equals cosine similarity
faiss.normalize_L2(embeddings)

# Build FAISS index using inner product (cosine similarity after normalization)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f'FAISS index built: {index.ntotal} vectors')

# Save index and chunks to disk
faiss.write_index(index, 'reviews.index')
with open('chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)
print('Saved: reviews.index, chunks.pkl')


In [ ]:
def search(query, k=5):
    """Find the most similar reviews for a given query using FAISS."""
    # Encode the query into a vector
    q_vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)

    # Search the index
    distances, indices = index.search(q_vec, k)

    print(f"\nQuery: '{query}'")
    print('=' * 60)
    for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), 1):
        print(f'\n[{rank}] Similarity: {score:.3f}')
        print(chunks[idx])
        print('-' * 60)


# Run demo queries to verify the index works correctly
search('dog food with good quality and taste')
search('terrible product, waste of money')
search('best coffee ever')


In [ ]:
import pickle

# Save Score and ProductId for each chunk
# These are used later for star rating and product ID filters in the app
metadata = df_sample[['Score', 'ProductId']].to_dict('records')

with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print(f'Metadata saved: {len(metadata)} records')
print(f'Sample: {metadata[0]}')
print(f'Score distribution: {df_sample["Score"].value_counts().sort_index().to_dict()}')


In [ ]:
# The Gradio app is saved as app.py
# Run it from the terminal:
#   GEMINI_API_KEY=your_key python app.py
#
# Or launch directly from here:
import subprocess, sys
subprocess.Popen([sys.executable, 'app.py'])
print('App starting at http://127.0.0.1:7860')
